# SatQuery AI — Google Colab GeoChat-7B GPU Server

Runs **real 4-bit GeoChat-7B** on a free Colab T4 GPU and tunnels it to your local backend.

### Instructions:
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order (Steps 1-5)
3. Copy the `trycloudflare.com` URL printed in Step 5
4. Local terminal: `python verify_dev1_dev2.py --geochat-url https://xxxx.trycloudflare.com`
   OR set `GEOCHAT_API_URL=https://xxxx.trycloudflare.com` in `backend/.env`

In [ ]:
# Step 1: Verify GPU (must show T4 or A100)
!nvidia-smi

In [ ]:
# Step 2: Clone GeoChat + install pinned dependencies.
# We force-reinstall pinned versions AFTER pip install -e . so GeoChat's
# setup.py cannot bump transformers and break the Bloom MPT module.
import os
if not os.path.isdir('GeoChat'):
    !git clone https://github.com/mbzuai-oryx/GeoChat.git
%cd GeoChat
!pip install -q -e .
!pip install -q --force-reinstall \
    "transformers==4.31.0" \
    "accelerate==0.21.0" \
    "bitsandbytes==0.41.0" \
    "einops==0.6.1" \
    "timm==0.6.13" \
    "pydantic==1.10.21"
!pip install -q fastapi uvicorn pillow httpx
print("\n[OK] All dependencies installed at pinned versions.")

In [ ]:
# Step 3: Patch GeoChat MPT module — fixes ImportError:
#   cannot import _expand_mask / _make_causal_mask from transformers.models.bloom
# These private symbols were removed in transformers >= 4.36.
# We remove those broken imports and inject pure-PyTorch shim functions.
import pathlib, re

p = pathlib.Path('geochat/model/language_model/mpt/hf_prefixlm_converter.py')
if p.exists():
    src = p.read_text(encoding='utf-8')
    # Remove the two broken import lines
    src = re.sub(
        r'from transformers\.models\.bloom\.modeling_bloom import _expand_mask as _expand_mask_bloom\n',
        '', src)
    src = re.sub(
        r'from transformers\.models\.bloom\.modeling_bloom import _make_causal_mask as _make_causal_mask_bloom\n',
        '', src)
    # Prepend pure-PyTorch shims
    shim = (
        "# --- SatQuery AI Bloom shim (compatible with all transformers versions) ---\n"
        "import torch as _ts\n"
        "def _expand_mask_bloom(mask, dtype, tgt_len=None):\n"
        "    bsz, src_len = mask.size()\n"
        "    tgt = tgt_len if tgt_len is not None else src_len\n"
        "    exp = mask[:, None, None, :].expand(bsz, 1, tgt, src_len).to(dtype)\n"
        "    return (1.0 - exp) * _ts.finfo(dtype).min\n"
        "def _make_causal_mask_bloom(shape, dtype, device, past_key_values_length=0):\n"
        "    bsz, tgt = shape\n"
        "    m = _ts.full((tgt, tgt), _ts.finfo(dtype).min, device=device)\n"
        "    c = _ts.arange(m.size(-1), device=device)\n"
        "    m.masked_fill_(c < (c + 1).view(m.size(-1), 1), 0)\n"
        "    m = m.to(dtype)\n"
        "    if past_key_values_length > 0:\n"
        "        m = _ts.cat([_ts.zeros(tgt, past_key_values_length, dtype=dtype, device=device), m], dim=-1)\n"
        "    return m[None, None, :, :].expand(bsz, 1, tgt, tgt + past_key_values_length)\n"
        "# --- end shim ---\n\n"
    )
    p.write_text(shim + src, encoding='utf-8')
    print('[OK] hf_prefixlm_converter.py patched — Bloom shims injected.')
else:
    print('[SKIP] hf_prefixlm_converter.py not found.')

In [ ]:
# Step 4: Load GeoChat-7B with 4-bit quantization (~4.5 GB VRAM, fits on T4)
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from geochat.model.builder import load_pretrained_model
from geochat.mm_utils import get_model_name_from_path
from geochat.conversation import Chat, conv_templates

print("\nLoading 4-bit GeoChat-7B from mbzuai-oryx/GeoChat ...")
model_path = "mbzuai-oryx/GeoChat"
model_name = get_model_name_from_path(model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=model_name,
    load_8bit=False,
    load_4bit=True,
    device="cuda"
)
chat = Chat(model, image_processor, tokenizer, device="cuda")
print("\n[OK] GeoChat-7B loaded successfully on GPU!")

In [ ]:
# Step 5: FastAPI server + Cloudflare Tunnel
# The tunnel gives you a free public HTTPS URL (no account needed).
import base64, io, threading, time
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="GeoChat Colab Server")

class ChatRequest(BaseModel):
    image_base64: str
    query: str

@app.get("/health")
def health_check():
    return {"status": "ok", "model": "GeoChat-7B-4bit",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}

@app.post("/chat")
def chat_endpoint(req: ChatRequest):
    raw = base64.b64decode(req.image_base64)
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    conv = conv_templates["llava_v1"].copy()
    img_list = []
    chat.upload_img(img, conv, img_list)
    chat.ask(req.query, conv)
    if img_list and not isinstance(img_list[0], torch.Tensor):
        chat.encode_img(img_list)
    streamer = chat.stream_answer(
        conv=conv, img_list=img_list,
        temperature=0.2, max_new_tokens=500, max_length=2000,
    )
    return {"response": "".join(streamer).strip()}

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()
time.sleep(2)
print("[OK] Server running on :8000")
print("     Quick test: curl http://127.0.0.1:8000/health")

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("\n=== COPY THE URL BELOW => paste in: python verify_dev1_dev2.py --geochat-url <URL> ===")
!cloudflared tunnel --url http://127.0.0.1:8000